#### MetaQuest3 wrist attach + collision-aware arm IK -> real `FFW SG2` + 에피소드 녹화

`real_MetaQuest3_sg2_teleop.ipynb` 와 같은 텔레옵에 녹화를 붙인 버전.

- Orin: `worker_outbound_meta` (관절 :5560 + 카메라 :5570/5571/5572)
- 컨트롤 창 `Record` 줄: `STOP` → `REC` 녹화 시작, `REC` → `STOP` 저장, `DEL` 녹화 중이면 버리고 아니면 마지막 에피소드 삭제
- 창 제목에 녹화 상태 표시
- 저장: `~/aiworker_data/<TASK_NAME>/episode_NNN/` — `joints.npz`(로봇 상태) / `action.npz`(텔레옵 명령) / `rgb_<cam>.mp4` / `rgb_<cam>_t.npy` / `meta.json`

In [11]:
import os
if "__vsc_ipynb_file__" in globals(): os.chdir(os.path.dirname(__vsc_ipynb_file__))   # VS Code 커널 cwd 보정
%run ../package/init_project.py

import json
from pathlib import Path
import numpy as np

from ri_motion_v5_package.init_scripts.init_ipython_setup import *
from ri_motion_v5_package.init_scripts.init_qt import *
from ri_motion_v5_package.mujoco_sim import *
from ri_motion_v5_package.kinematics import *
from ri_motion_v5_package.utility import *
from ri_motion_v5_package.qt import *
from utils import *
from package.episode_recorder import *   # not `from episode_recorder`: PACKAGE_DIR is on sys.path only during %run

app = get_qt_app()



[ffw_sg2_vr_teleoperation] initialized.
 - PROJECT_DIR: /Users/yujin/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation


In [12]:
native_backend = build_sg2_collision_aware_ik_backend(
    clean   = False,
    verbose = True,
    required = True,)
print("bind info:",native_backend.native_info())


[build_native]
  module      : _mujoco_collision_aware_ik_bind
  output      : mujoco_sim/so/_mujoco_collision_aware_ik_bind.cpython-310-darwin.so
  compiler    : clang++
  project cpp:
    [01] mujoco_sim/cpp/bindings.cpp
    [02] mujoco_sim/cpp/signed_distance_bindings.cpp
    [03] mujoco_sim/cpp/signed_distance.cpp
    [04] mujoco_sim/cpp/ik_bindings.cpp
    [05] mujoco_sim/cpp/ik.cpp
  project hpp:
    [01] mujoco_sim/cpp/signed_distance_bindings.hpp
    [02] mujoco_sim/cpp/signed_distance.hpp
    [03] mujoco_sim/cpp/ik_bindings.hpp
    [04] mujoco_sim/cpp/ik.hpp
    [05] mujoco_sim/cpp/mujoco_handles.hpp
  defines:
    [01] RI_NATIVE_ENABLE_SIGNED_DISTANCE
    [02] RI_NATIVE_ENABLE_IK
    [03] RI_NATIVE_MODULE_NAME=_mujoco_collision_aware_ik_bind
  nanobind cpp:
    [01] nanobind/src/nb_combined.cpp
  commands    : hidden; pass show_commands=True to print full commands
[build_native] compile
  [01/06] mujoco_sim/cpp/bindings.cpp -> build/native/bindings.o
  [02/06] mujoco_sim/cpp/

In [13]:
RENDER_PORT = 5556
PUB_TO_SG2_PORT = 5561
PUB_TO_SG2_IP = get_ip_address()
SUB_FROM_SG2_PORT = 5560
SUB_FROM_SG2_IP = "192.168.6.2"

ip_address_metaquest3 = "192.168.6.100"
metaquest3_port = 5555
ALLOW_MISSING_ROBOT_STATE = False

# Recording
CAMERAS   = {"head": 5570, "wrist_left": 5571, "wrist_right": 5572}
SAVE_ROOT = Path.home() / "aiworker_data"
TASK_NAME = "pick_and_place"
RECORD_HZ = 30
MAX_SEC   = 120        # 에피소드당 안전 타임아웃

socket_sub = create_zmq_sub_socket(
    ip_address                 = ip_address_metaquest3,
    port                       = metaquest3_port,
    wait_seconds_after_connect = 0,)

print("PUB_TO_SG2_IP",PUB_TO_SG2_IP)
print(f"MetaQuest3: tcp://{ip_address_metaquest3}:{metaquest3_port}")


PUB_TO_SG2_IP None
MetaQuest3: tcp://192.168.6.100:5555


In [14]:
xml_path = merge_mjcfs(
    included_mjcf_files = [
        str(FLOOR_XML_PATH),
        str(SG2_XML_PATH),
    ],
    output_xml_path = str(SG2_SCENE_XML_PATH),)
env = MuJoCoParser(rel_xml_path=xml_path,verbose=False)


[merge_mjcfs] Merging [2] MJCF files:
  - [0] [/Users/yujin/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/floor_simple_white.xml]
  - [1] [/Users/yujin/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/ffw_sg2.xml]
[merge_mjcfs] Saved merged XML to:[/Users/yujin/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/scene_ffw_sg2_metaquest3.xml]
MuJoCo model loaded from [/Users/yujin/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/scene_ffw_sg2_metaquest3.xml].


In [ ]:
env.reset()
env.set_p("base_link","body",(0.0,0.0,0.01))

real_command_joint_names = (
    list(SG2_REAL_COMMAND_JOINT_NAMES)
    + [SG2_GRIPPER_DRIVER_JOINT_NAMES[hand_type] for hand_type in SG2_HAND_TYPES]
)
command_joint_names = filter_existing_joint_names(env,real_command_joint_names)
q_msg_format_dict = {
    "lift"      : {"prefixes":["lift_joint"]},
    "head"      : {"prefixes":["head_joint"]},
    "arm_l"     : {"prefixes":["arm_l_joint"]},
    "arm_r"     : {"prefixes":["arm_r_joint"]},
    "gripper_l" : {"prefixes":["gripper_l_joint1"]},
    "gripper_r" : {"prefixes":["gripper_r_joint1"]},
}
q_msg_layout = build_q_msg_layout(
    env               = env,
    q_msg_format_dict = q_msg_format_dict,
    use_joint_names   = command_joint_names,)
q_use_min = env.get_joint_mins(command_joint_names)
q_use_max = env.get_joint_maxs(command_joint_names)

print("Joint message layout")
print("--------------------")
for q_msg_key in q_msg_layout["q_msg_keys"]:
    joint_names = q_msg_layout["qmsgkeys_to_usejointnames_dict"][q_msg_key]
    joint_idxs = q_msg_layout["qmsgkeys_to_usejointidxs_dict"][q_msg_key]
    print("%-8s | %2d joints | idxs:%s | %s"%(q_msg_key,len(joint_names),joint_idxs,joint_names))
print("--------------------")
print("%-8s | %2d joints"%("total",len(command_joint_names)))
print("unused:",q_msg_layout["unused_joint_names"])

try:
    q_use,recv_count = get_latest_robot_qpos(
        env             = env,
        q_msg_layout    = q_msg_layout,
        use_joint_names = command_joint_names,
        sub_ip          = SUB_FROM_SG2_IP,
        sub_port        = SUB_FROM_SG2_PORT,
        timeout_sec     = 1.0,
        verbose         = True,)
    print("[Robot init] OK. recv_count:[%d]"%recv_count)
except Exception:
    if not ALLOW_MISSING_ROBOT_STATE:
        raise
    q_use = env.get_qpos(joint_names=command_joint_names)
    print("[Robot init] missing robot state; using current MuJoCo q because ALLOW_MISSING_ROBOT_STATE=True.")

q_use = np.clip(q_use,q_use_min,q_use_max)
env.forward(q=q_use,joint_names=command_joint_names)


Joint message layout
--------------------
lift     |  1 joints | idxs:[0] | ['lift_joint']
head     |  2 joints | idxs:[1, 2] | ['head_joint1', 'head_joint2']
arm_l    |  7 joints | idxs:[3, 4, 5, 6, 7, 8, 9] | ['arm_l_joint1', 'arm_l_joint2', 'arm_l_joint3', 'arm_l_joint4', 'arm_l_joint5', 'arm_l_joint6', 'arm_l_joint7']
arm_r    |  7 joints | idxs:[10, 11, 12, 13, 14, 15, 16] | ['arm_r_joint1', 'arm_r_joint2', 'arm_r_joint3', 'arm_r_joint4', 'arm_r_joint5', 'arm_r_joint6', 'arm_r_joint7']
gripper_l |  1 joints | idxs:[17] | ['gripper_l_joint1']
gripper_r |  1 joints | idxs:[18] | ['gripper_r_joint1']
--------------------
total    | 19 joints
unused: []
[ZMQ Subscriber] pid:[73074] endpoint:[tcp://192.168.6.2:5560]
[ZMQ] role:[sub] ip:[192.168.6.2] port:[5560] hwm:[1] conflate:[True]
[ZMQ SUB] pid:[73074] Socket closed.


UnpicklingError: invalid load key, '{'.

: 

In [ ]:
input_hz = 20
robot_hz = 100
render_hz = 20
max_missing_quest_ticks = quest_missing_tick_limit(
    input_hz              = input_hz,
    missing_grace_seconds = 1.0,)

hand_types = list(SG2_HAND_TYPES)
hand_labels = dict(SG2_HAND_LABELS)

collision_geom_pairs,collision_candidate_info = create_sg2_collision_candidate_geom_pairs(
    env               = env,
    init_exclude_dist = 0.05,)
print_sg2_collision_candidate_summary(collision_candidate_info)

collision_ik_solvers,arm_joint_names = create_sg2_collision_aware_arm_ik_solvers(
    env                = env,
    mujoco_parser_cls  = MuJoCoParser,
    body_ik_solver_cls = BodyPositionIKSolverRevPriBase,
    enable_depenetration = False,)
active_ik_solver = collision_ik_solvers["tracking"]
collision_ik_result = make_idle_collision_ik_result(active_ik_solver,safe_margin=0.05)

T_quest_start = env.get_T(name="head_link2",type="body")
quest_retargeting = QuestPalmRetargeting(start_T=T_quest_start)

T_quest_pose_by_idx = {}
T_virtual_quest_pose_by_idx = {}
T_quest_wrist = {hand_type:None for hand_type in hand_types}
T_raw_wrist_target = {hand_type:None for hand_type in hand_types}
T_attach_wrist_target = {hand_type:None for hand_type in hand_types}
T_virtual_quest_wrist = {hand_type:None for hand_type in hand_types}
T_sg2_wrist = {hand_type:None for hand_type in hand_types}

quest_missing_ticks = {hand_type:max_missing_quest_ticks+1 for hand_type in hand_types}
quest_all_missing_ticks = max_missing_quest_ticks+1
quest_attach_state = {
    hand_type:create_quest_wrist_attach_state(
        hand_type             = hand_type,
        attach_seconds        = 2.5,
        attach_start_distance = 0.20,)
    for hand_type in hand_types
}
last_solved_wrist_target = {hand_type:None for hand_type in hand_types}
arm_ik_info_by_hand = {hand_type:None for hand_type in hand_types}
wrist_target_motion = {hand_type:(np.inf,np.inf) for hand_type in hand_types}
gripper_driver_joint_names = {
    hand_type:get_sg2_gripper_joint_names(hand_type,driver_only=True)
    for hand_type in hand_types
}
gripper_q_by_hand = {
    hand_type:float(env.get_qpos(joint_names=gripper_driver_joint_names[hand_type])[0])
    for hand_type in hand_types
}
gripper_info_by_hand = {hand_type:{} for hand_type in hand_types}
detach_gesture_state = create_sg2_detach_gesture_state(
    hand_types   = hand_types,
    finger_names = ("middle","ring","little"),)
render_arm_ik_targets = []

ctrl_radio = MultiRadioQtWidget(
    title          = "SG2 MetaQuest3 Command",
    x_offset       = 0.00,
    y_offset       = 0.00,
    window_width   = 0.24,
    window_height  = 0.14,
    label_texts    = ["Robot command","Record"],
    option_texts   = [["OFF","ON"],["STOP","REC","DEL"]],
    initial_texts  = ["OFF","STOP"],)

pub_robot = ZMQPubSub(role="pub",port=PUB_TO_SG2_PORT,ip=PUB_TO_SG2_IP,verbose=True)
pub_render = ZMQPubSub(role="pub",port=RENDER_PORT,verbose=True)

sub_joint = ZMQPubSub(role="sub",ip=SUB_FROM_SG2_IP,port=SUB_FROM_SG2_PORT,verbose=True)
sub_cams = open_camera_subs(SUB_FROM_SG2_IP,CAMERAS)
recorder = EpisodeRecorder(
    sub_joint    = sub_joint,
    sub_cams     = sub_cams,
    task_dir     = SAVE_ROOT/TASK_NAME,
    hz           = RECORD_HZ,
    max_sec      = MAX_SEC,
    action_names = command_joint_names,
    meta         = dict(task=TASK_NAME,robot_ip=SUB_FROM_SG2_IP,teleop="metaquest3"),)
print("[rec] save to:",SAVE_ROOT/TASK_NAME,"| next:",next_episode_dir(SAVE_ROOT/TASK_NAME).name)

print("ZMQ publishers")
print("--------------")
print("Robot    : tcp://*:%d"%PUB_TO_SG2_PORT)
print("Renderer pub : tcp://*:%d"%RENDER_PORT)
print("Render view  : run real_render_MetaQuest3_sg2.ipynb in another kernel/window")
for hand_type in hand_types:
    print("%s arm IK joints:"%hand_type,len(arm_joint_names[hand_type]),arm_joint_names[hand_type])
print("collision IK joints:",active_ik_solver.joints_use)
print("Depenetration IK: disabled.")
print("Quest thumb-index pinch -> SG2 gripper: enabled.")
print("Detach gesture: bend middle/ring/little on both hands.")


Collision candidates
--------------------
robot collision geom:[34]
collision geom:[35]
candidate geom pairs:[493]
excluded initial geom pairs:[37]
MuJoCo model loaded from [/Users/yujin/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/scene_ffw_sg2_metaquest3.xml].
[ZMQ Publisher] pid:[50912] endpoint:[tcp://0.0.0.0:5561]
[ZMQ] role:[pub] ip:[None] port:[5561] hwm:[1] conflate:[True]
[ZMQ Publisher] pid:[50912] endpoint:[tcp://0.0.0.0:5556]
[ZMQ] role:[pub] ip:[None] port:[5556] hwm:[1] conflate:[True]
[ZMQ Subscriber] pid:[50912] endpoint:[tcp://192.168.6.2:5560]
[ZMQ] role:[sub] ip:[192.168.6.2] port:[5560] hwm:[1] conflate:[True]
[camera] head        ≈20 Hz, (376, 672, 3)
[camera] wrist_left  ≈19 Hz, (240, 424, 3)
[camera] wrist_right ≈18 Hz, (240, 424, 3)
[camera] 사용: ['head', 'wrist_left', 'wrist_right']
[rec] save to: /Users/yujin/aiworker_data/pick_and_place | next: episode_002
ZMQ publishers
--------------
Robot    : tcp://*:5561
Renderer pub : tcp://*:5556
Re

In [ ]:
tmr_input = SimpleTimer(name="MetaQuest3",Hz=input_hz,verbose=True)
tmr_robot = SimpleTimer(name="Robot Pub",Hz=robot_hz,verbose=True)
tmr_render = SimpleTimer(name="Render Pub",Hz=render_hz,verbose=True)
tmr_record = SimpleTimer(name="Record",Hz=RECORD_HZ,verbose=True)
tmr_input.start()
tmr_robot.start()
tmr_render.start()
tmr_record.start()

robot_pub_hz_actual = 0.0
prev_on_flag = False
prev_rec_opt = "STOP"
publish_ok = False

try:
    while ctrl_radio.isVisible():
        app.processEvents()

        on_flag = ctrl_radio.get_button_option(label_text="Robot command") == "ON"
        if on_flag != prev_on_flag:
            print("[Robot Pub] %s"%("ON" if on_flag else "OFF"))
            prev_on_flag = on_flag

        rec_opt = ctrl_radio.get_button_option(label_text="Record")
        if rec_opt != prev_rec_opt:
            if rec_opt == "REC":
                recorder.start()
            elif rec_opt == "STOP":
                recorder.stop()
            elif rec_opt == "DEL":
                recorder.discard_or_delete_last()
                ctrl_radio.set_button_option(label_text="Record",option_text="STOP")
                rec_opt = "STOP"
            prev_rec_opt = rec_opt

        if tmr_input.do_run():
            # Receive MetaQuest3
            T_quest_wrist,T_quest_pose_by_idx,quest_missing_ticks,quest_all_missing_ticks,quest_updated,_ = update_metaquest3_wrist_data(
                socket_sub              = socket_sub,
                quest_retargeting       = quest_retargeting,
                hand_types              = hand_types,
                T_quest_wrist           = T_quest_wrist,
                T_quest_pose_by_idx     = T_quest_pose_by_idx,
                quest_missing_ticks     = quest_missing_ticks,
                quest_all_missing_ticks = quest_all_missing_ticks,
                quest_attach_state      = quest_attach_state,
                check_distance          = 0.10,)

            # Attach Quest wrist to SG2 wrist
            arm_ik_info_by_hand = {hand_type:None for hand_type in hand_types}
            (
                T_raw_wrist_target,
                T_attach_wrist_target,
                T_virtual_quest_wrist,
                attached_wrist_targets,
                quest_attach_state,
                last_solved_wrist_target,
                wrist_target_motion,
            ) = update_sg2_wrist_attach_targets(
                env                      = env,
                hand_types               = hand_types,
                T_quest_wrist            = T_quest_wrist,
                quest_missing_ticks      = quest_missing_ticks,
                quest_updated            = quest_updated,
                quest_attach_state       = quest_attach_state,
                last_solved_wrist_target = last_solved_wrist_target,
                max_missing_quest_ticks  = max_missing_quest_ticks,
                dt                       = getattr(tmr_input,"dt",1.0/input_hz),
                attach_seconds           = 3.0,
                attach_start_distance    = 0.15,)

            # Middle/ring/little bend on both hands -> detach
            detach_gesture_state = update_sg2_detach_gesture_state(
                T_pose_by_idx           = T_quest_pose_by_idx,
                hand_types              = hand_types,
                quest_missing_ticks     = quest_missing_ticks,
                max_missing_quest_ticks = max_missing_quest_ticks,
                dt                      = getattr(tmr_input,"dt",1.0/input_hz),
                detach_gesture_state    = detach_gesture_state,
                finger_names            = ("middle","ring","little"),
                enter_bend_ratio        = 0.45,
                exit_bend_ratio         = 0.28,
                mean_enter_bend_ratio   = 0.40,
                mean_exit_bend_ratio    = 0.25,
                min_closed_finger_count = 2,
                hold_seconds_to_detach  = 1.2,)
            if detach_gesture_state["should_detach"]:
                quest_attach_state = apply_sg2_detach_gesture_to_attach_state(
                    quest_attach_state = quest_attach_state,
                    hand_types         = hand_types,)
                attached_wrist_targets = {}
                T_attach_wrist_target = {hand_type:None for hand_type in hand_types}
                T_virtual_quest_wrist = {hand_type:None for hand_type in hand_types}
                last_solved_wrist_target = {hand_type:None for hand_type in hand_types}

            # Collision-aware arm IK
            collision_ik_result = solve_sg2_collision_aware_wrist_ik(
                env                    = env,
                tracking_solver        = collision_ik_solvers["tracking"],
                depenetration_solver   = None,
                wrist_targets_by_hand  = attached_wrist_targets,
                collision_geom_pairs   = collision_geom_pairs,
                native_backend         = native_backend,
                safe_margin            = 0.05,
                signed_distance_topk   = 5,
                enable_depenetration   = False,)
            active_ik_solver = collision_ik_result["solver"]
            last_solved_wrist_target,arm_ik_info_by_hand = update_last_solved_wrist_targets(
                last_solved_wrist_target = last_solved_wrist_target,
                arm_ik_info_by_hand      = arm_ik_info_by_hand,
                attached_wrist_targets   = attached_wrist_targets,
                ik_info                  = collision_ik_result["ik_info"],)

            # Quest thumb-index pinch -> SG2 gripper
            gripper_q_by_hand,gripper_info_by_hand = update_sg2_gripper_from_quest_pinch(
                env                     = env,
                T_pose_by_idx           = T_quest_pose_by_idx,
                hand_types              = hand_types,
                quest_missing_ticks     = quest_missing_ticks,
                max_missing_quest_ticks = max_missing_quest_ticks,
                prev_gripper_q_by_hand  = gripper_q_by_hand,
                pinch_closed_distance   = 0.025,
                pinch_open_distance     = 0.095,
                smoothing_alpha         = 0.35,)

            q_use = np.clip(env.get_qpos(joint_names=command_joint_names),q_use_min,q_use_max)
            env.forward(q=q_use,joint_names=command_joint_names)

            T_virtual_quest_pose_by_idx = {}
            if quest_all_missing_ticks <= max_missing_quest_ticks:
                T_virtual_quest_pose_by_idx = virtualize_quest_pose_dicts_from_wrists(
                    T_pose_by_idx = T_quest_pose_by_idx,
                    T_quest_wrist = T_quest_wrist,
                    T_quest_target = T_attach_wrist_target,
                    hand_types     = hand_types,)
            T_sg2_wrist = {
                hand_type:get_sg2_wrist_control_T(env,hand_type)
                for hand_type in hand_types
            }
            render_arm_ik_targets = get_solver_target_infos(active_ik_solver,env)
            tmr_input.end()

        attached = {
            hand_type:bool(quest_attach_state[hand_type]["is_attached"])
            for hand_type in hand_types
        }
        publish_ok = on_flag and any(attached.values())

        if tmr_robot.do_run():
            if publish_ok:
                robot_pub_hz_actual = 1.0/tmr_robot.sec_elps_diff if tmr_robot.sec_elps_diff > 0 else 0.0
                pub_robot.send(obj=get_qmsg_from_qpos(qpos=q_use,q_msg_layout=q_msg_layout))
            else:
                robot_pub_hz_actual = 0.0
            tmr_robot.end()

        if tmr_render.do_run():
            send_render_qpos(
                pub_render          = pub_render,
                qpos                = q_use,
                joint_names         = command_joint_names,
                q_msg_layout        = q_msg_layout,
                msg_type            = "real_metaquest3_wrist_ffw_sg2",
                on_flag             = publish_ok,
                robot_hz            = robot_hz,
                render_hz           = render_hz,
                robot_pub_hz_actual = robot_pub_hz_actual,
                extra               = make_sg2_real_render_extra(
                    hand_types                    = hand_types,
                    raw_robot_command_on          = on_flag,
                    attached                      = attached,
                    quest_attach_state            = quest_attach_state,
                    quest_missing_ticks           = quest_missing_ticks,
                    quest_all_missing_ticks       = quest_all_missing_ticks,
                    max_missing_quest_ticks       = max_missing_quest_ticks,
                    collision_ik_result           = collision_ik_result,
                    T_quest_start                 = T_quest_start,
                    T_quest_pose_by_idx           = T_quest_pose_by_idx,
                    T_virtual_quest_pose_by_idx   = T_virtual_quest_pose_by_idx,
                    T_quest_wrist                 = T_quest_wrist,
                    T_virtual_quest_wrist         = T_virtual_quest_wrist,
                    T_raw_wrist_target            = T_raw_wrist_target,
                    T_attach_wrist_target         = T_attach_wrist_target,
                    T_sg2_wrist                   = T_sg2_wrist,
                    active_arm_ik_targets         = render_arm_ik_targets,
                    arm_ik_info_by_hand           = arm_ik_info_by_hand,
                    wrist_target_motion           = wrist_target_motion,
                    detach_gesture_state          = detach_gesture_state,
                    plot_quest_nodes              = False,
                    plot_arm_ik_targets           = False,),)
            tmr_render.end()

        if tmr_record.do_run():
            recorder.poll(action_q=q_use,action_on=publish_ok)
            if prev_rec_opt == "REC" and not recorder.recording:   # timeout 으로 자동 저장된 경우
                ctrl_radio.set_button_option(label_text="Record",option_text="STOP")
                prev_rec_opt = "STOP"
            ctrl_radio.setWindowTitle(recorder.status())
            tmr_record.end()

except KeyboardInterrupt:
    print("Interrupted.")
finally:
    pub_render.close()
    pub_robot.close()
    recorder.close()
    sub_joint.close()
    ctrl_radio.close()
    close_zmq_socket(socket_sub)
    print("Done.")


[MetaQuest3] Initialized with [20.0]HZ. MAX_SEC:[inf]sec.
[Robot Pub] Initialized with [100.0]HZ. MAX_SEC:[inf]sec.
[Render Pub] Initialized with [20.0]HZ. MAX_SEC:[inf]sec.
[Record] Initialized with [30.0]HZ. MAX_SEC:[inf]sec.
[MetaQuest3] Start ([20.0]HZ. MAX_SEC:[inf]sec).
[Robot Pub] Start ([100.0]HZ. MAX_SEC:[inf]sec).
[Render Pub] Start ([20.0]HZ. MAX_SEC:[inf]sec).
[Record] Start ([30.0]HZ. MAX_SEC:[inf]sec).
[Robot Pub][218][2.2s] delayed! T:[10.0]ms interval:[11.0]ms prev_run:[0.0]ms.
[Robot Pub][3638][36.4s] delayed! T:[10.0]ms interval:[11.4]ms prev_run:[0.0]ms.
[Robot Pub][3706][37.1s] delayed! T:[10.0]ms interval:[18.4]ms prev_run:[0.0]ms.
[Render Pub][742][37.1s] delayed! T:[50.0]ms interval:[60.1]ms prev_run:[0.1]ms.
[Robot Pub][3707][37.1s] delayed! T:[10.0]ms interval:[15.3]ms prev_run:[0.0]ms.
[MetaQuest3][744][37.3s] delayed! T:[50.0]ms interval:[239.4]ms prev_run:[1.5]ms.
[Robot Pub][3713][37.3s] delayed! T:[10.0]ms interval:[229.0]ms prev_run:[0.0]ms.
[Render Pub][

#### 수집 결과 확인

In [ ]:
eps = list_episodes(SAVE_ROOT/TASK_NAME)
print(f"{len(eps)} episodes in {SAVE_ROOT/TASK_NAME}")
tot = 0.0
for e in eps:
    m = json.loads((e/"meta.json").read_text()); tot += m["duration_sec"]
    nf = m["n_frames"] if isinstance(m["n_frames"],dict) else {"rgb":m["n_frames"]}
    print(f"  {e.name}: {m['duration_sec']:5.1f}s  joints {m['n_joint_samples']:5d}  action {m.get('n_action_samples',0):5d}  frames "
          + "  ".join(f"{c} {v}" for c,v in nf.items()))
print(f"total {tot/60:.1f} min")